# COMPX310 Lab 10
Alexis Manosca 1644633  
Ammia Camia 1607516

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

# ID as random seed
ID = 1607516

# Ammi's import
df = pd.read_csv('data.csv')

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

# Checking for missing or duplicated rows
print("\nMissing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

# Preprocessing the data
filename_col = "file"
target_col = "target"

# Handle missing target values
# df = df.dropna(subset=[target_col])

# Split the target and features
filenames = df[filename_col].copy()
X = df.drop(columns=[filename_col, target_col])
y = df[target_col].astype(int)

print("\nFeatures:", X.shape)
print("Target:", y.shape)
print("Number of classes:", (y.nunique()))
print("\nClass distribution:")
print(y.value_counts().sort_index())

# Cross validation code
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=ID)

# To store each model's out-of-fold probabilities for the error analysis
oof_probs = {}

# Define class names once to be reused
class_names = ["mountain", "street", "buildings", "sea", "forest", "glacier"]

Dataset shape: (14034, 2050)

Column names:
['a0', 'a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7', 'a8', 'a9', 'a10', 'a11', 'a12', 'a13', 'a14', 'a15', 'a16', 'a17', 'a18', 'a19', 'a20', 'a21', 'a22', 'a23', 'a24', 'a25', 'a26', 'a27', 'a28', 'a29', 'a30', 'a31', 'a32', 'a33', 'a34', 'a35', 'a36', 'a37', 'a38', 'a39', 'a40', 'a41', 'a42', 'a43', 'a44', 'a45', 'a46', 'a47', 'a48', 'a49', 'a50', 'a51', 'a52', 'a53', 'a54', 'a55', 'a56', 'a57', 'a58', 'a59', 'a60', 'a61', 'a62', 'a63', 'a64', 'a65', 'a66', 'a67', 'a68', 'a69', 'a70', 'a71', 'a72', 'a73', 'a74', 'a75', 'a76', 'a77', 'a78', 'a79', 'a80', 'a81', 'a82', 'a83', 'a84', 'a85', 'a86', 'a87', 'a88', 'a89', 'a90', 'a91', 'a92', 'a93', 'a94', 'a95', 'a96', 'a97', 'a98', 'a99', 'a100', 'a101', 'a102', 'a103', 'a104', 'a105', 'a106', 'a107', 'a108', 'a109', 'a110', 'a111', 'a112', 'a113', 'a114', 'a115', 'a116', 'a117', 'a118', 'a119', 'a120', 'a121', 'a122', 'a123', 'a124', 'a125', 'a126', 'a127', 'a128', 'a129', 'a130', 'a131', 'a132', '

## Logistic Regression

In [2]:
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(solver="lbfgs", max_iter=3000, random_state=ID))])

param_grid_lr = {
    "model__C": [0.001, 0.01, 0.1, 1, 10],
    "model__penalty": ["l2"]
}

lr_grid = GridSearchCV(lr_pipeline, param_grid_lr, cv=cv, scoring="accuracy", n_jobs=-1, verbose=1)
lr_grid.fit(X, y)

# Table of every combination tried
lr_results = pd.DataFrame(lr_grid.cv_results_)
lr_table = lr_results[["param_model__C", "param_model__penalty", "mean_test_score", "std_test_score"]]
lr_table = lr_table.sort_values("mean_test_score", ascending=False)
display(lr_table)

print("Best params:", lr_grid.best_params_)
print("Best cv accuracy:", lr_grid.best_score_)

# Out-of-fold predictions for the best setup, using the same folds
best_lr = lr_grid.best_estimator_

lr_probs = cross_val_predict(best_lr, X, y, cv=cv, method="predict_proba", n_jobs=-1)
lr_preds = lr_probs.argmax(axis=1)

# Save for the error analyis later
oof_probs["logistic"] = lr_probs

# Sanity check: these two should be almost the same
print("OOF accuracy:", (lr_preds == y).mean(), "| CV best score:", lr_grid.best_score_)

# Confusion matrix and classification report
print("Confusion matrix:")
print(confusion_matrix(y, lr_preds))

print("\nClassification report:")
print(classification_report(y, lr_preds, target_names=class_names))

Fitting 10 folds for each of 5 candidates, totalling 50 fits


c:\Users\KES\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,param_model__C,param_model__penalty,mean_test_score,std_test_score
0,0.001,l2,0.942782,0.006500
1,0.010,l2,0.937437,0.005093
2,0.100,l2,0.930953,0.004494
3,1.000,l2,0.928459,0.006325
4,10.000,l2,0.924327,0.004471


Best params: {'model__C': 0.001, 'model__penalty': 'l2'}
Best cv accuracy: 0.9427816461672485
OOF accuracy: 0.9427818155907083 | CV best score: 0.9427816461672485
Confusion matrix:
[[2335    4    5   18    4  146]
 [   6 2246  124    4    2    0]
 [   3  126 2050    8    2    2]
 [  22    2    9 2228    2   11]
 [  20    1    4    3 2242    1]
 [ 218    6   14   27    9 2130]]

Classification report:
              precision    recall  f1-score   support

    mountain       0.90      0.93      0.91      2512
      street       0.94      0.94      0.94      2382
   buildings       0.93      0.94      0.93      2191
         sea       0.97      0.98      0.98      2274
      forest       0.99      0.99      0.99      2271
     glacier       0.93      0.89      0.91      2404

    accuracy                           0.94     14034
   macro avg       0.94      0.94      0.94     14034
weighted avg       0.94      0.94      0.94     14034



## k-Nearest Neighbour

In [3]:
from sklearn.neighbors import KNeighborsClassifier

# Define pipeline with scaling
knn_pipeline = Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier())])

param_grid_knn = {
    "model__n_neighbors": [3, 5, 7, 11, 15, 25],
    "model__weights": ["uniform", "distance"],
    "model__metric": ["euclidean", "manhattan", "cosine"]
}

knn_grid = GridSearchCV(knn_pipeline, param_grid_knn, cv=cv, scoring="accuracy", n_jobs=-1, verbose=1)
knn_grid.fit(X, y)

# Table of every combination tried
knn_results = pd.DataFrame(knn_grid.cv_results_)
knn_table = knn_results[["param_model__n_neighbors", "param_model__weights", "param_model__metric", "mean_test_score", "std_test_score"]]
knn_table = knn_table.sort_values("mean_test_score", ascending=False)
display(knn_table)

print("Best params:", knn_grid.best_params_)
print("Best cv accuracy:", knn_grid.best_score_)

# Out-of-fold predictions for the best setup, using the same folds
best_knn = knn_grid.best_estimator_

knn_probs = cross_val_predict(best_knn, X, y, cv=cv, method="predict_proba")
knn_preds = knn_probs.argmax(axis=1)

# Save for the error analysis later
oof_probs["knn"] = knn_probs

# Sanity check: these two should be almost the same
print("OOF accuracy:", (knn_preds == y).mean(), "| CV best score:", knn_grid.best_score_)

# Confusion matrix and classification report
print("Confusion matrix:")
print(confusion_matrix(y, knn_preds))

print("\nClassification report:")
print(classification_report(y, knn_preds, target_names=class_names))

Fitting 10 folds for each of 36 candidates, totalling 360 fits


,param_model__n_neighbors,param_model__weights,param_model__metric,mean_test_score,std_test_score
31,11,distance,cosine,0.928959,0.006540
17,7,distance,manhattan,0.928460,0.006646
30,11,uniform,cosine,0.928104,0.006355
18,11,uniform,manhattan,0.927889,0.006737
19,11,distance,manhattan,0.927818,0.006610
16,7,uniform,manhattan,0.927676,0.006295
33,15,distance,cosine,0.927462,0.006298
34,25,uniform,cosine,0.927177,0.005690
27,5,distance,cosine,0.927106,0.006533
35,25,distance,cosine,0.927034,0.005807


Best params: {'model__metric': 'cosine', 'model__n_neighbors': 11, 'model__weights': 'distance'}
Best cv accuracy: 0.9289589564892486
OOF accuracy: 0.9289582442639305 | CV best score: 0.9289589564892486
Confusion matrix:
[[2269    7    4   23    4  205]
 [   4 2236  129    5    8    0]
 [   4  212 1967    7    1    0]
 [  24    8   14 2216    3    9]
 [  38    6    4    4 2218    1]
 [ 218    9   10   24   12 2131]]

Classification report:
              precision    recall  f1-score   support

    mountain       0.89      0.90      0.90      2512
      street       0.90      0.94      0.92      2382
   buildings       0.92      0.90      0.91      2191
         sea       0.97      0.97      0.97      2274
      forest       0.99      0.98      0.98      2271
     glacier       0.91      0.89      0.90      2404

    accuracy                           0.93     14034
   macro avg       0.93      0.93      0.93     14034
weighted avg       0.93      0.93      0.93     14034



## Random Forest

In [4]:
from sklearn.ensemble import RandomForestClassifier

# Grid search with the same 10-fold cross validation
param_grid_rf = {
    "n_estimators": [100, 300],
    "max_depth": [None, 20],
    "max_features": ["sqrt", "log2"],
    "min_samples_leaf": [1, 2],
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=ID, n_jobs=-1),
    param_grid=param_grid_rf,
    cv=cv,
    scoring="accuracy",
    verbose=1,
)
rf_grid.fit(X, y)

# Table of every combination tried
rf_results = pd.DataFrame(rf_grid.cv_results_)
rf_table = rf_results[["param_n_estimators", "param_max_depth",
                       "param_max_features", "param_min_samples_leaf",
                       "mean_test_score", "std_test_score"]]
rf_table = rf_table.sort_values("mean_test_score", ascending=False)
display(rf_table)

print("Best params:", rf_grid.best_params_)
print("Best cv accuracy:", rf_grid.best_score_)

# Out-of-fold predictions for the best setup
best_rf = rf_grid.best_estimator_

rf_probs = cross_val_predict(best_rf, X, y, cv=cv, method="predict_proba")
rf_preds = np.argmax(rf_probs, axis=1)

# Save for the error analysis later
oof_probs["Random Forest"] = rf_probs

# Sanity check: these two should be almost the same
print("OOF accuracy:", (rf_preds == y).mean(), "| CV best score:", rf_grid.best_score_)

# Confusion matrix and classification report
print("\nConfusion matrix:")
print(confusion_matrix(y, rf_preds))

print("\nClassification report:")
print(classification_report(y, rf_preds, target_names=class_names))

Fitting 10 folds for each of 16 candidates, totalling 160 fits


,param_n_estimators,param_max_depth,param_max_features,param_min_samples_leaf,mean_test_score,std_test_score
1,300,None,sqrt,1,0.932664,0.005331
9,300,20,sqrt,1,0.932451,0.006534
11,300,20,sqrt,2,0.932165,0.005222
3,300,None,sqrt,2,0.931453,0.006686
13,300,20,log2,1,0.931381,0.006635
15,300,20,log2,2,0.931239,0.004580
7,300,None,log2,2,0.930597,0.004689
2,100,None,sqrt,2,0.930384,0.006294
5,300,None,log2,1,0.930099,0.007471
10,100,20,sqrt,2,0.929101,0.005717


Best params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 300}
Best cv accuracy: 0.932663827817071
OOF accuracy: 0.9326635314236853 | CV best score: 0.932663827817071

Confusion matrix:
[[2369    4    2   22    6  109]
 [   6 2229  138    4    5    0]
 [   5  172 2002    9    3    0]
 [  22    2   15 2228    1    6]
 [  27    2   10    5 2227    0]
 [ 286    3   15   49   17 2034]]

Classification report:
              precision    recall  f1-score   support

    mountain       0.87      0.94      0.91      2512
      street       0.92      0.94      0.93      2382
   buildings       0.92      0.91      0.92      2191
         sea       0.96      0.98      0.97      2274
      forest       0.99      0.98      0.98      2271
     glacier       0.95      0.85      0.89      2404

    accuracy                           0.93     14034
   macro avg       0.93      0.93      0.93     14034
weighted avg       0.93      0.93      0.93     14034



## XGBoost

In [5]:
from xgboost import XGBClassifier

# Grid search with the same 10-fold cross validation
param_grid_xgb = {
    "n_estimators": [100, 300],
    "max_depth": [3, 5],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8]
}

xgb_grid = GridSearchCV(
    estimator=XGBClassifier(objective="multi:softprob", num_class=6, eval_metric="mlogloss", random_state=ID, n_jobs=-1),
    param_grid=param_grid_xgb,
    cv=cv,
    scoring="accuracy",
    verbose=1,
)

xgb_grid.fit(X, y)

# Table of every combination tried
xgb_results = pd.DataFrame(xgb_grid.cv_results_)
xgb_table = xgb_results[["param_n_estimators", "param_max_depth",
                         "param_learning_rate", "param_subsample",
                         "mean_test_score", "std_test_score"]]
xgb_table = xgb_table.sort_values("mean_test_score", ascending=False)
display(xgb_table)

print("Best params:", xgb_grid.best_params_)
print("Best cv accuracy:", xgb_grid.best_score_)

# Out-of-fold predictions for the best setup
best_xgb = xgb_grid.best_estimator_

xgb_probs = cross_val_predict(best_xgb, X, y, cv=cv, method="predict_proba")
xgb_preds = np.argmax(xgb_probs, axis=1)

# Save for the error analysis later
oof_probs["XGBoost"] = xgb_probs

# Sanity check: these two should be almost the same
print("OOF accuracy:", (xgb_preds == y).mean(), "| CV best score:", xgb_grid.best_score_)

# Confusion matrix and classification report
print("\nConfusion matrix:")
print(confusion_matrix(y, xgb_preds))

print("\nClassification report:")
print(classification_report(y, xgb_preds, target_names=class_names))


Fitting 10 folds for each of 8 candidates, totalling 80 fits


,param_n_estimators,param_max_depth,param_learning_rate,param_subsample,mean_test_score,std_test_score
7,300,5,0.10,0.8,0.940003,0.005996
5,300,3,0.10,0.8,0.939505,0.004817
6,100,5,0.10,0.8,0.935514,0.005865
4,100,3,0.10,0.8,0.933805,0.005456
3,300,5,0.01,0.8,0.924898,0.005086
1,300,3,0.01,0.8,0.914067,0.006796
2,100,5,0.01,0.8,0.912856,0.007690
0,100,3,0.01,0.8,0.894614,0.008660


Best params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.8}
Best cv accuracy: 0.9400034622593424
OOF accuracy: 0.9400028502208921 | CV best score: 0.9400034622593424

Confusion matrix:
[[2355    3    4   19    6  125]
 [   6 2246  125    2    3    0]
 [   5  145 2031    8    1    1]
 [  24    4   11 2221    2   12]
 [  21    3    5    4 2237    1]
 [ 244   10    9   27   12 2102]]

Classification report:
              precision    recall  f1-score   support

    mountain       0.89      0.94      0.91      2512
      street       0.93      0.94      0.94      2382
   buildings       0.93      0.93      0.93      2191
         sea       0.97      0.98      0.98      2274
      forest       0.99      0.99      0.99      2271
     glacier       0.94      0.87      0.91      2404

    accuracy                           0.94     14034
   macro avg       0.94      0.94      0.94     14034
weighted avg       0.94      0.94      0.94     14034



## Fully Connected Neural Network (FCNN)

In [6]:
from sklearn.neural_network import MLPClassifier

# Defines the pipeline - scales the data first then goes into the neural network
nn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        max_iter=300,
        early_stopping=True,
        random_state=ID,
    )),
])

# Defines the grid
param_grid_nn = {
    "mlp__hidden_layer_sizes": [(128,), (256,), (128, 64)],
    "mlp__alpha": [0.0001, 0.001, 0.01], # Regularisation strength
    "mlp__epsilon": [1e-8, 1e-6, 1e-4],
}

# Grid search - tries every combination and runs cross validation for each one
nn_grid = GridSearchCV(
    estimator=nn_pipeline,
    param_grid=param_grid_nn,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)
nn_grid.fit(X, y)

# Display the results
nn_results = pd.DataFrame(nn_grid.cv_results_)
nn_table = nn_results[["param_mlp__hidden_layer_sizes", "param_mlp__alpha",
                       "param_mlp__epsilon", "mean_test_score", "std_test_score"]]
nn_table = nn_table.sort_values("mean_test_score", ascending=False)
display(nn_table)

print("Best params:", nn_grid.best_params_)
print("Best cv accuracy:", nn_grid.best_score_)

# Display the best model and its predictions
best_nn = nn_grid.best_estimator_

nn_probs = cross_val_predict(best_nn, X, y, cv=cv, method="predict_proba")
nn_preds = np.argmax(nn_probs, axis=1)

# Save for the error analysis later
oof_probs["Neural Network"] = nn_probs

# Sanity check: these two should be almost the same
print("OOF accuracy:", (nn_preds == y).mean(), "| CV best score:", nn_grid.best_score_)

# Display confusion matrix and classification report
print("\nConfusion matrix:")
print(confusion_matrix(y, nn_preds))

print("\nClassification report:")
print(classification_report(y, nn_preds, target_names=class_names))

Fitting 10 folds for each of 27 candidates, totalling 270 fits


,param_mlp__hidden_layer_sizes,param_mlp__alpha,param_mlp__epsilon,mean_test_score,std_test_score
22,"(256,)",0.0100,1.000000e-06,0.938150,0.005793
4,"(256,)",0.0001,1.000000e-06,0.938007,0.005731
6,"(128,)",0.0001,1.000000e-04,0.937865,0.007014
0,"(128,)",0.0001,1.000000e-08,0.937795,0.006828
21,"(128,)",0.0100,1.000000e-06,0.937794,0.006012
3,"(128,)",0.0001,1.000000e-06,0.937723,0.006511
18,"(128,)",0.0100,1.000000e-08,0.937723,0.005890
19,"(256,)",0.0100,1.000000e-08,0.937722,0.006760
8,"(128, 64)",0.0001,1.000000e-04,0.937580,0.004627
17,"(128, 64)",0.0010,1.000000e-04,0.937295,0.004302


Best params: {'mlp__alpha': 0.01, 'mlp__epsilon': 1e-06, 'mlp__hidden_layer_sizes': (256,)}
Best cv accuracy: 0.9381496305231158
OOF accuracy: 0.9381502066410147 | CV best score: 0.9381496305231158

Confusion matrix:
[[2326    4    4   21    5  152]
 [   3 2237  133    5    2    2]
 [   4  130 2043   10    2    2]
 [  24    4   15 2212    1   18]
 [  19    0    2    2 2248    0]
 [ 253    7   10   25    9 2100]]

Classification report:
              precision    recall  f1-score   support

    mountain       0.88      0.93      0.90      2512
      street       0.94      0.94      0.94      2382
   buildings       0.93      0.93      0.93      2191
         sea       0.97      0.97      0.97      2274
      forest       0.99      0.99      0.99      2271
     glacier       0.92      0.87      0.90      2404

    accuracy                           0.94     14034
   macro avg       0.94      0.94      0.94     14034
weighted avg       0.94      0.94      0.94     14034



## My Conclusions

**What we learnt**

---

**Best parameter combination found**

---

**Observation about `C`**

---

**One thing that surprised us or differed from initial prediction**

---

**One limitation of this experiment**

---
